# Détection de fraude : modèles non supervisés (version corrigée, MLflow intégré)

**Ce que fait ce notebook** : Isolation Forest, One-Class SVM et LOF, tous tracés dans MLflow avec le même schéma que le notebook supervisé. Les deux familles sont donc comparables dans le tableau de bord.

**Corrections par rapport à la version précédente**
1. **MLflow intégré** : le notebook n'enregistrait rien dans MLflow.
2. **Balayage de `contamination` sans réentraîner** : cette valeur ne change pas les arbres, seulement le seuil d'alerte. Avant, le modèle était réentraîné 5 fois pour obtenir des PR-AUC et ROC-AUC identiques. Maintenant : un entraînement par modèle, puis un seuil par valeur.
3. **Seuil calibré sans le test** : le seuil vient d'un quantile des scores sur un jeu de validation temporel (sans labels). Avant, le seuil dépendait de scores calculés sur les données d'entraînement.
4. **Noms de métriques unifiés** : `PR_AUC`, `ROC_AUC`, `F1` (avant : `PR-AUC` / `F1-score` selon les cellules).
5. **Même découpage et mêmes variables que le notebook supervisé**, avec les mêmes garde-fous contre les variables post-transaction.
6. **One-Class SVM et LOF sous-échantillonnés** (`OCSVM_MAX_TRAIN`, `LOF_MAX_TRAIN`) : leur coût devient prohibitif sur plusieurs millions de lignes.
7. **Nettoyage** : suppression de `%whos`, des cellules de réimport et des sauvegardes CSV dispersées (tout est dans MLflow).

**Remarque** : ces modèles n'utilisent pas les labels pour apprendre. Comparez-les au supervisé surtout sur `PR_AUC` et le rappel au budget d'alertes, pas sur `F1` qui dépend de `contamination`.

In [ ]:
from pathlib import Path

NOTEBOOK_NAME = "unsupervised_fraud_detection"

# ---------- Données
DATA_PATH = Path("../data/processed/fraud_eda_features.parquet")
DATASET_VERSION = "paysim_v1"       # à changer si les données ou le prétraitement changent

# ---------- MLflow
TRACKING_URI = "sqlite:///mlflow.db"
EXPERIMENT_NAME = "Fraud_Detection_Experiments"
LOG_MODELS = True                   # False = plus rapide, n'enregistre pas les pipelines
REPLACE_PREVIOUS_RUNS = True        # anciens runs de même clé -> corbeille MLflow (restaurable)

# ---------- Découpage temporel (identique dans les deux notebooks)
RANDOM_STATE = 42
TEST_FRACTION = 0.20
VAL_FRACTION_OF_TRAIN = 0.15        # sert à calibrer le seuil d'alerte, SANS utiliser les labels
USE_RAW_STEP = False
TOP_K_FRACTIONS = [0.001, 0.005, 0.01]

# ---------- Spécifique à ce notebook
CONTAMINATION_GRID = [0.001, 0.002, 0.003, 0.005, 0.01]   # part de transactions signalées
REFERENCE_CONTAMINATION = 0.01      # run "de référence" de chaque modèle (celui qui est comparé)
FIT_ON_NORMAL_ONLY = False          # True : entraîne uniquement sur les transactions normales (utilise les labels)
OCSVM_MAX_TRAIN = 50_000            # One-Class SVM : coût en O(n²), il faut sous-échantillonner
LOF_MAX_TRAIN = 200_000             # LOF : idem, avec n_neighbors=20

# ---------- Test rapide (vérifier que tout tourne avant le vrai run)
QUICK_RUN = False
QUICK_RUN_ROWS = 200_000
if QUICK_RUN:
    OCSVM_MAX_TRAIN, LOF_MAX_TRAIN = 5_000, 20_000

In [ ]:
# ============================================================
# SOCLE COMMUN : cellule identique dans les deux notebooks.
# Elle fixe le "contrat" MLflow lu par le tableau de bord :
# mêmes tags, mêmes noms de métriques, mêmes artefacts.
# ============================================================
import tempfile
import time
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.tracking import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

# ---------- Variables : uniquement des informations connues AVANT la transaction
TARGET = "isFraud"
CATEGORICAL_FEATURES = ["type"]
NUMERIC_FEATURES = [
    "amount", "oldbalanceOrg", "oldbalanceDest",
    "amount_nul", "ratioOrig_nul", "ratioDest_nul", "hour_of_day",
]
if USE_RAW_STEP:
    NUMERIC_FEATURES = ["step"] + NUMERIC_FEATURES
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
# colonnes calculées APRÈS la transaction (fuite de données si utilisées)
POST_TRANSACTION_PATTERNS = ("newbalance", "ecart")


# ---------------------------------------------------------------
# Données et découpage temporel : fit (entraînement) / val / test
# ---------------------------------------------------------------
def load_dataset():
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"Fichier introuvable : {DATA_PATH.resolve()}")
    df = pd.read_parquet(DATA_PATH)
    if QUICK_RUN and len(df) > QUICK_RUN_ROWS:
        df = df.sample(QUICK_RUN_ROWS, random_state=RANDOM_STATE)
    df = df.sort_values("step", kind="mergesort").reset_index(drop=True)
    df["hour_of_day"] = df["step"] % 24          # 1 step = 1 heure dans PaySim
    missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
    if missing:
        raise KeyError(f"Colonnes manquantes dans le parquet : {missing}")
    for col in NUMERIC_FEATURES:
        if pd.api.types.is_bool_dtype(df[col]):
            df[col] = df[col].astype("int8")
    return df


def audit_features(df):
    leaking = [c for c in FEATURES if any(p in c.lower() for p in POST_TRANSACTION_PATTERNS)]
    if leaking:
        raise ValueError(f"Variables post-transaction détectées dans FEATURES : {leaking}")
    excluded = [c for c in df.columns if c not in FEATURES]
    print("Variables utilisées :", FEATURES)
    print("Variables exclues   :", excluded)
    return "pre_transaction"


def temporal_split(df):
    cut_test = int(len(df) * (1 - TEST_FRACTION))
    step_test = df["step"].iloc[cut_test]
    train_period = df[df["step"] < step_test]
    test_df = df[df["step"] >= step_test]

    cut_val = int(len(train_period) * (1 - VAL_FRACTION_OF_TRAIN))
    step_val = train_period["step"].iloc[cut_val]
    fit_df = train_period[train_period["step"] < step_val]
    val_df = train_period[train_period["step"] >= step_val]

    for name, part in [("fit", fit_df), ("val", val_df), ("test", test_df)]:
        if part.empty or part[TARGET].sum() == 0:
            raise ValueError(f"Le jeu '{name}' est vide ou ne contient aucune fraude.")
    assert fit_df["step"].max() < val_df["step"].min()
    assert val_df["step"].max() < test_df["step"].min()
    info = {"step_val_start": int(step_val), "step_test_start": int(step_test)}
    return fit_df.copy(), val_df.copy(), test_df.copy(), info


def describe_splits(**parts):
    rows = []
    for name, part in parts.items():
        rows.append({
            "jeu": name, "lignes": len(part),
            "fraudes": int(part[TARGET].sum()),
            "taux_fraude_%": round(part[TARGET].mean() * 100, 4),
            "step_min": int(part["step"].min()), "step_max": int(part["step"].max()),
        })
    return pd.DataFrame(rows)


def build_preprocessor():
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric, NUMERIC_FEATURES),
        ("cat", categorical, CATEGORICAL_FEATURES),
    ])


# ---------------------------------------------------------------
# Métriques
# ---------------------------------------------------------------
def _frac_name(f):
    return f"{f * 100:g}".replace(".", "p")


def ranking_metrics(y_true, scores):
    # métriques indépendantes du seuil : PR_AUC, ROC_AUC et rappel / précision au top-k
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    m = {
        "PR_AUC": average_precision_score(y_true, scores),
        "ROC_AUC": roc_auc_score(y_true, scores),
    }
    y_sorted = y_true[np.argsort(-scores, kind="mergesort")]
    n, n_pos = len(y_true), int(y_true.sum())
    for f in TOP_K_FRACTIONS:
        k = max(1, int(round(n * f)))
        tp_k = int(y_sorted[:k].sum())
        m[f"Recall_top_{_frac_name(f)}pct"] = tp_k / n_pos if n_pos else 0.0
        m[f"Precision_top_{_frac_name(f)}pct"] = tp_k / k
    return m


def threshold_metrics(y_true, scores, threshold, amounts=None, suffix=""):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    m = {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Alerts": int(tp + fp),
        "Alert_rate": (tp + fp) / len(y_true),
    }
    if amounts is not None:
        amounts = np.asarray(amounts, dtype=float)
        fraud_total = amounts[y_true == 1].sum()
        caught = amounts[(y_true == 1) & (y_pred == 1)].sum()
        m["Amount_Recall"] = float(caught / fraud_total) if fraud_total > 0 else 0.0
    return {f"{k}{suffix}": v for k, v in m.items()}, y_pred


def threshold_table(y_true, scores, n_low=60, n_high=140):
    # table seuil -> TP/FP/FN/TN/précision/rappel, pour le curseur de seuil du tableau de bord
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    order = np.argsort(s, kind="mergesort")
    s_sorted, y_sorted = s[order], y[order]
    cum_pos = np.concatenate([[0], np.cumsum(y_sorted)])
    n, n_pos = len(y), int(y.sum())
    qs = np.concatenate([np.linspace(0, 0.99, n_low, endpoint=False), np.linspace(0.99, 1.0, n_high)])
    thr = np.unique(np.quantile(s, qs))
    idx = np.searchsorted(s_sorted, thr, side="left")
    flagged = n - idx
    tp = n_pos - cum_pos[idx]
    fp = flagged - tp
    fn = n_pos - tp
    tn = (n - n_pos) - fp
    precision = np.divide(tp, flagged, out=np.zeros(len(tp), dtype=float), where=flagged > 0)
    recall = tp / n_pos if n_pos else np.zeros(len(tp))
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros(len(tp), dtype=float), where=denom > 0)
    return pd.DataFrame({
        "threshold": thr, "flagged": flagged, "alert_rate": flagged / n,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Precision": precision, "Recall": recall, "F1": f1,
    })


def _downsample(x, y, n=500):
    if len(x) <= n:
        return x, y
    idx = np.unique(np.linspace(0, len(x) - 1, n).astype(int))
    return x[idx], y[idx]


def curve_frames(y_true, scores):
    p, r, _ = precision_recall_curve(y_true, scores)
    r_d, p_d = _downsample(r, p)
    fpr, tpr, _ = roc_curve(y_true, scores)
    f_d, t_d = _downsample(fpr, tpr)
    return (pd.DataFrame({"recall": r_d, "precision": p_d}),
            pd.DataFrame({"fpr": f_d, "tpr": t_d}))


def plot_confusion(y_true, y_pred, title):
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, labels=[0, 1], display_labels=["Normal", "Fraude"],
        ax=ax, cmap="Blues", values_format=",d",
    )
    ax.set_title(title)
    fig.tight_layout()
    return fig


def plot_pr(pr_df, title, pr_auc):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(pr_df["recall"], pr_df["precision"])
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{title} (PR-AUC = {pr_auc:.4f})")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


# ---------------------------------------------------------------
# MLflow
# ---------------------------------------------------------------
mlflow.set_tracking_uri(TRACKING_URI)
_experiment = mlflow.set_experiment(EXPERIMENT_NAME)
EXPERIMENT_ID = _experiment.experiment_id
client = MlflowClient()
BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")


def archive_previous_runs(run_key):
    # envoie à la corbeille MLflow (restaurable) les anciens runs de même clé :
    # une ré-exécution du notebook ne crée donc pas de doublons dans le tableau de bord
    if not REPLACE_PREVIOUS_RUNS:
        return 0
    old = client.search_runs([EXPERIMENT_ID], filter_string=f"tags.run_key = '{run_key}'", max_results=1000)
    for r in old:
        client.delete_run(r.info.run_id)
    return len(old)


def _log_model(model, x_example):
    kwargs = dict(serialization_format="cloudpickle", input_example=x_example)
    try:                                              # MLflow >= 3
        mlflow.sklearn.log_model(model, name="model", **kwargs)
    except TypeError:                                 # MLflow 2.x
        mlflow.sklearn.log_model(model, artifact_path="model", **kwargs)


def log_full_run(*, run_name, tags, params, thresholds, y_true, scores, amounts=None,
                 extra_metrics=None, model=None, x_example=None, importance=None, title=""):
    # thresholds : {suffixe: seuil}. Le suffixe "" correspond au seuil principal.
    y_true = np.asarray(y_true).astype(int)
    metrics = dict(ranking_metrics(y_true, scores))
    preds = {}
    for suffix, thr in thresholds.items():
        m, y_pred = threshold_metrics(y_true, scores, thr, amounts, suffix)
        metrics.update(m)
        preds[suffix] = y_pred
    if extra_metrics:
        metrics.update(extra_metrics)

    pr_df, roc_df = curve_frames(y_true, scores)
    table = threshold_table(y_true, scores)
    n_archived = archive_previous_runs(tags["run_key"])

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({**tags, "batch_id": BATCH_ID, "dataset_version": DATASET_VERSION,
                         "notebook": NOTEBOOK_NAME})
        mlflow.log_params({k: str(v)[:250] for k, v in params.items()})
        mlflow.log_metrics({k: float(v) for k, v in metrics.items()
                            if v is not None and np.isfinite(v)})

        with tempfile.TemporaryDirectory() as tmp:
            tmp = Path(tmp)

            def _log(path):
                mlflow.log_artifact(str(path), artifact_path="evaluation")

            for suffix, y_pred in preds.items():
                label = suffix or "_seuil_principal"
                fig = plot_confusion(y_true, y_pred, f"{title} | {label.strip('_')}")
                p = tmp / f"confusion_matrix{suffix}.png"
                fig.savefig(p, dpi=150)
                plt.close(fig)
                _log(p)
                cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                p = tmp / f"confusion_matrix{suffix}.csv"
                pd.DataFrame(cm, index=["Actual_Normal", "Actual_Fraud"],
                             columns=["Predicted_Normal", "Predicted_Fraud"]).to_csv(p)
                _log(p)

            fig = plot_pr(pr_df, title, metrics["PR_AUC"])
            p = tmp / "pr_curve.png"
            fig.savefig(p, dpi=150)
            plt.close(fig)
            _log(p)

            for name, frame in [("pr_curve_points", pr_df), ("roc_curve_points", roc_df),
                                ("threshold_table", table)]:
                p = tmp / f"{name}.csv"
                frame.to_csv(p, index=False)
                _log(p)

            if importance is not None:
                p = tmp / "feature_importance.csv"
                importance.to_csv(p, index=False)
                _log(p)

        if model is not None and LOG_MODELS:
            _log_model(model, x_example)

    if n_archived:
        print(f"   ({n_archived} ancien(s) run(s) de même clé mis à la corbeille MLflow)")
    return run.info.run_id, metrics


# ---------------------------------------------------------------
# Lecture du classement depuis MLflow (même fonction que le tableau de bord)
# ---------------------------------------------------------------
LEADERBOARD_TAGS = ["family", "model", "strategy", "purpose", "contamination",
                    "is_reference", "feature_set", "batch_id", "dataset_version"]
LEADERBOARD_METRICS = [
    "PR_AUC", "ROC_AUC", "Precision", "Recall", "F1", "MCC", "Amount_Recall", "Alert_rate",
    "Recall_top_0p1pct", "Precision_top_0p1pct", "Recall_top_0p5pct", "Recall_top_1pct",
    "Precision_tuned", "Recall_tuned", "F1_tuned",
    "Training_time_sec", "Prediction_time_sec",
]


def load_leaderboard(dataset_version=None, only_reference=False):
    dv = dataset_version or DATASET_VERSION
    runs = mlflow.search_runs(
        experiment_ids=[EXPERIMENT_ID],
        filter_string=f"tags.dataset_version = '{dv}'",
        output_format="pandas",
    )
    if runs.empty:
        return pd.DataFrame()

    def col(key):
        return runs[key] if key in runs.columns else pd.Series(np.nan, index=runs.index)

    out = pd.DataFrame({"run_id": runs["run_id"], "run_name": col("tags.mlflow.runName")})
    for t in LEADERBOARD_TAGS:
        out[t] = col(f"tags.{t}")
    for m in LEADERBOARD_METRICS:
        out[m] = col(f"metrics.{m}")
    if only_reference:
        out = out[out["is_reference"] == "true"]
    return out.sort_values("PR_AUC", ascending=False).reset_index(drop=True)


print(f"MLflow : {TRACKING_URI} | expérience '{EXPERIMENT_NAME}' (id={EXPERIMENT_ID}) | batch {BATCH_ID}")

## 1. Chargement, audit des variables et découpage temporel

In [ ]:
df = load_dataset()
FEATURE_SET = audit_features(df)
fit_df, val_df, test_df, split_info = temporal_split(df)

if FIT_ON_NORMAL_ONLY:
    fit_df = fit_df[fit_df[TARGET] == 0]

X_fit = fit_df[FEATURES]
X_val = val_df[FEATURES]
X_test, y_test = test_df[FEATURES], test_df[TARGET].astype("int8")
test_amounts = test_df["amount"].to_numpy()

print()
display(describe_splits(fit=fit_df, val=val_df, test=test_df))
print("Seuils temporels :", split_info)
print("Les labels ne servent QU'À l'évaluation sur le test (le seuil est calibré sur le score de val, sans labels).")

## 2. Détecteurs

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM


def make_detectors():
    # (estimateur, taille max d'entraînement ou None)
    return {
        "Isolation Forest": (IsolationForest(
            n_estimators=150, max_samples=256, random_state=RANDOM_STATE, n_jobs=-1), None),
        "One-Class SVM": (OneClassSVM(
            kernel="rbf", gamma="scale", nu=0.01, cache_size=1000), OCSVM_MAX_TRAIN),
        "LOF": (LocalOutlierFactor(
            n_neighbors=20, novelty=True, metric="minkowski", n_jobs=-1), LOF_MAX_TRAIN),
    }

## 3. Entraînement, balayage de la contamination et suivi MLflow

Un run par couple (modèle, contamination). Le modèle n'est enregistré qu'une fois par modèle, sur le run de référence.

In [ ]:
# Principe : `contamination` ne change PAS le modèle appris, seulement le seuil d'alerte.
# On entraîne donc UNE fois par modèle, on calcule les scores UNE fois,
# puis on déduit un seuil par valeur de contamination (quantile des scores de validation).
all_results, trained_models, test_scores = [], {}, {}

for model_name, (estimator, max_train) in make_detectors().items():
    print(f"\n{'=' * 70}\n{model_name}")

    X_model = X_fit
    if max_train is not None and len(X_fit) > max_train:
        X_model = X_fit.sample(max_train, random_state=RANDOM_STATE)
        print(f"Sous-échantillon d'entraînement : {len(X_model):,} lignes")

    pipe = Pipeline([("preprocessor", build_preprocessor()), ("model", estimator)])

    t0 = time.time()
    pipe.fit(X_model)
    train_time = time.time() - t0

    t1 = time.time()
    score_test = -pipe.decision_function(X_test)      # plus le score est élevé, plus c'est anormal
    pred_time = time.time() - t1
    score_val = -pipe.decision_function(X_val)
    test_scores[model_name] = score_test

    for c in CONTAMINATION_GRID:
        thr = float(np.quantile(score_val, 1 - c))
        is_ref = bool(np.isclose(c, REFERENCE_CONTAMINATION))

        params = {
            "algorithm": model_name, "strategy": "unsupervised", "feature_set": FEATURE_SET,
            "features": ",".join(FEATURES), "contamination": c, "score_threshold": round(thr, 6),
            "fit_size": len(X_model), "val_size": len(X_val), "test_size": len(X_test),
            "n_fraud_test": int(y_test.sum()), "fit_on_normal_only": FIT_ON_NORMAL_ONLY,
            "random_state": RANDOM_STATE, "use_raw_step": USE_RAW_STEP,
            "step_val_start": split_info["step_val_start"], "step_test_start": split_info["step_test_start"],
            **{f"model_{k}": v for k, v in estimator.get_params(deep=False).items()},
        }
        tags = {
            "family": "unsupervised", "model": model_name, "strategy": "unsupervised",
            "purpose": "contamination_sweep", "feature_set": FEATURE_SET,
            "contamination": str(c), "is_reference": str(is_ref).lower(),
            "run_key": f"unsupervised|{model_name}|c={c}|{DATASET_VERSION}",
        }

        run_id, metrics = log_full_run(
            run_name=f"{model_name.replace(' ', '_')}_c{c}",
            tags=tags, params=params, thresholds={"": thr},
            y_true=y_test, scores=score_test, amounts=test_amounts,
            extra_metrics={"Training_time_sec": train_time, "Prediction_time_sec": pred_time},
            model=pipe if is_ref else None,          # le modèle est identique : on ne le stocke qu'une fois
            x_example=X_model.head(5), title=f"{model_name} (contamination {c})",
        )
        all_results.append({"Model": model_name, "Contamination": c, "run_id": run_id, **metrics})

    ref = [r for r in all_results if r["Model"] == model_name]
    print(f"PR-AUC {ref[0]['PR_AUC']:.4f} | ROC-AUC {ref[0]['ROC_AUC']:.4f} | "
          f"entraînement {train_time:.1f}s | scoring test {pred_time:.1f}s")
    trained_models[model_name] = pipe

results_df = pd.DataFrame(all_results)

## 4. Résultats

In [ ]:
cols = ["Model", "Contamination", "PR_AUC", "ROC_AUC", "Precision", "Recall", "F1", "MCC",
        "Alert_rate", "Amount_Recall", "Recall_top_0p1pct", "Recall_top_0p5pct", "Recall_top_1pct"]
display(results_df[cols].round(4))

In [ ]:
# 1. Impact de la contamination sur chaque modèle
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for ax, (model_name, grp) in zip(axes, results_df.groupby("Model", sort=False)):
    for metric in ["Precision", "Recall", "F1"]:
        ax.plot(grp["Contamination"], grp[metric], marker="o", label=metric)
    ax.set(title=model_name, xlabel="Contamination")
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("Score")
fig.tight_layout()
plt.show()

# 2. Courbes PR et ROC superposées (indépendantes du seuil)
ref = results_df.drop_duplicates("Model").set_index("Model")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for model_name, scores in test_scores.items():
    p, r, _ = precision_recall_curve(y_test, scores)
    fpr, tpr, _ = roc_curve(y_test, scores)
    r_d, p_d = _downsample(r, p)
    f_d, t_d = _downsample(fpr, tpr)
    axes[0].plot(r_d, p_d, label=f"{model_name} (AP={ref.loc[model_name, 'PR_AUC']:.3f})")
    axes[1].plot(f_d, t_d, label=f"{model_name} (AUC={ref.loc[model_name, 'ROC_AUC']:.3f})")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall (test chronologique)")
axes[1].set(xlabel="Taux de faux positifs", ylabel="Taux de vrais positifs", title="ROC")
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
plt.show()

# 3. Budget d'alertes : part des fraudes retrouvées dans le top-k des scores
budget_labels = {f"Recall_top_{_frac_name(f)}pct": f"top {f * 100:g} %" for f in TOP_K_FRACTIONS}
ref[list(budget_labels)].rename(columns=budget_labels).plot.bar(
    figsize=(8, 4), rot=0, title="Rappel au budget d'alertes")
plt.grid(alpha=0.3, axis="y")
plt.show()

## 5. Classement lu depuis MLflow et export pour le tableau de bord

In [ ]:
# Classement lu directement dans MLflow : c'est exactement ce que lira le tableau de bord.
# Les runs supervisés apparaissent ici une fois l'autre notebook exécuté.
leaderboard = load_leaderboard(only_reference=True)
display(leaderboard.round(4).head(20))

Path("results").mkdir(exist_ok=True)
load_leaderboard().to_csv("results/leaderboard.csv", index=False)
print("Exporté : results/leaderboard.csv")
print("Interface MLflow : mlflow ui --backend-store-uri sqlite:///mlflow.db")

## Contrat MLflow pour le tableau de bord

Les deux notebooks écrivent dans la même expérience (`Fraud_Detection_Experiments`) avec le même schéma :

| Élément | Contenu |
|---|---|
| **Tags** | `family` (supervised / unsupervised), `model`, `strategy`, `purpose`, `feature_set`, `is_reference`, `dataset_version`, `batch_id`, `run_key`, `contamination` (non supervisé) |
| **Métriques sans seuil** | `PR_AUC`, `ROC_AUC`, `Recall_top_0p1pct`, `Precision_top_0p1pct`, `Recall_top_0p5pct`, `Recall_top_1pct`... |
| **Métriques au seuil principal** | `Precision`, `Recall`, `F1`, `MCC`, `TP`, `FP`, `FN`, `TN`, `Alerts`, `Alert_rate`, `Amount_Recall` |
| **Métriques au seuil optimisé** (supervisé) | mêmes noms avec le suffixe `_tuned`, plus `Tuned_threshold` |
| **Temps** | `Training_time_sec`, `Prediction_time_sec` |
| **Artefacts** (`evaluation/`) | `confusion_matrix*.png/.csv`, `pr_curve.png`, `pr_curve_points.csv`, `roc_curve_points.csv`, `threshold_table.csv`, `feature_importance.csv` |
| **Modèle** | pipeline complet (`model/`) |

`threshold_table.csv`, `pr_curve_points.csv` et `roc_curve_points.csv` permettent au tableau de bord de superposer les courbes et d'animer un curseur de seuil sans recharger les modèles.